# AI Proctor — Audio Classifier Retraining (Colab)

Browser-recorded `.webm` дээжүүдээс WavLM cheating classifier-ийг fine-tune хийнэ.

**Шаардлагатай зүйл:**
- Google Drive дээр `training_data.zip` файл (доорхи folder structure-тай):
  ```
  training_data/
    normal/*.webm
    talking/*.webm
    whispering/*.webm
    multiple/*.webm
  ```
- Эх `cheating_classifier_v2/` folder Drive дээр (~500 MB) — анхны model

**GPU runtime сонгох:** Runtime → Change runtime type → **T4 GPU**

## 1. Drive холбох + dependency суулгах

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers==4.44.2 accelerate==0.34.2 pydub librosa
!apt-get -qq install -y ffmpeg

## 2. Path-ыг тохируулах

Drive дээрх таны file-уудын path-ыг доор оруул:

In [ ]:
import os
from pathlib import Path

# ── ЗАСАХ — танай Drive дээрх path-ууд ──────────────────────────
DRIVE_BASE     = '/content/drive/MyDrive/ai-proctor'  # Drive дотрох folder
TRAIN_ZIP      = f'{DRIVE_BASE}/training_data.zip'    # Browser-аар бичсэн дээжүүд
SOURCE_MODEL   = f'{DRIVE_BASE}/models/cheating_classifier_v2'  # Анхны model
OUTPUT_MODEL   = f'{DRIVE_BASE}/models/cheating_classifier_v3'  # Гарах model
# ────────────────────────────────────────────────────────────────

# Local working dirs (Colab-н SSD-д хурдан)
WORK_DIR  = Path('/content/work')
WORK_DIR.mkdir(exist_ok=True)
TRAIN_DIR = WORK_DIR / 'training_data'
MODEL_DIR = WORK_DIR / 'source_model'
OUT_DIR   = WORK_DIR / 'output_model'

assert os.path.exists(TRAIN_ZIP), f'Олдсонгүй: {TRAIN_ZIP}'
assert os.path.exists(SOURCE_MODEL), f'Олдсонгүй: {SOURCE_MODEL}'
print('✓ Path-ууд зөв')

## 3. Training data + анхны model-ийг unpack хийх

In [ ]:
!unzip -q -o '{TRAIN_ZIP}' -d '{WORK_DIR}'
!cp -r '{SOURCE_MODEL}' '{MODEL_DIR}'

# Дээжний тоо шалгах
for cls in ['normal', 'talking', 'whispering', 'multiple']:
    path = TRAIN_DIR / cls
    count = len(list(path.glob('*.webm'))) if path.exists() else 0
    flag = '✓' if count >= 15 else '⚠'
    print(f'{flag} {cls:12} {count}')

## 4. Dataset + Training код

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    Trainer,
    TrainingArguments,
)
from pydub import AudioSegment

LABELS      = ['normal', 'talking', 'whispering', 'multiple']
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
PROD_MAP    = {
    'normal':     'normal',
    'talking':    'talking_to_someone',
    'multiple':   'multiple_voices',
    'whispering': 'whispering',
}
TARGET_SR  = 16000
TARGET_LEN = 5

def webm_to_pcm(webm_path):
    audio = AudioSegment.from_file(str(webm_path), format='webm')
    audio = audio.set_frame_rate(TARGET_SR).set_channels(1).set_sample_width(2)
    pcm = np.frombuffer(audio.raw_data, np.int16).astype(np.float32) / 32768.0
    target = TARGET_SR * TARGET_LEN
    return pcm[:target] if len(pcm) >= target else np.pad(pcm, (0, target - len(pcm)))

class WebMDataset(Dataset):
    def __init__(self, files, labels, extractor):
        self.files     = files
        self.labels    = labels
        self.extractor = extractor
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        pcm  = webm_to_pcm(self.files[idx])
        feat = self.extractor(pcm, sampling_rate=TARGET_SR, return_tensors='pt',
                              max_length=TARGET_SR * TARGET_LEN, truncation=True, padding='max_length')
        item = {k: v.squeeze(0) for k, v in feat.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *a, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print('✓ Класс/функцууд бэлэн')

## 5. Дээжийг цуглуулж train/val split

In [ ]:
files, labels = [], []
for lbl in LABELS:
    cls_dir = TRAIN_DIR / lbl
    if cls_dir.exists():
        for f in sorted(cls_dir.glob('*.webm')):
            files.append(f)
            labels.append(LABEL_TO_ID[lbl])

# Stratified 80/20 split
by_class = {i: [] for i in range(len(LABELS))}
for i, lbl_id in enumerate(labels):
    by_class[lbl_id].append(i)

rng = np.random.default_rng(42)
train_idx, val_idx = [], []
for cls_id, idxs in by_class.items():
    rng.shuffle(idxs)
    n_val = max(1, len(idxs) // 5)
    val_idx.extend(idxs[:n_val])
    train_idx.extend(idxs[n_val:])

train_files  = [files[i]  for i in train_idx]
train_labels = [labels[i] for i in train_idx]
val_files    = [files[i]  for i in val_idx]
val_labels   = [labels[i] for i in val_idx]

print(f'Train: {len(train_files)}  |  Val: {len(val_files)}')

# Class weights (imbalance-г засах)
counts = {l: sum(1 for x in labels if x == LABEL_TO_ID[l]) for l in LABELS}
total  = sum(counts.values())
weights = torch.tensor(
    [total / (len(LABELS) * counts[l]) if counts[l] > 0 else 0.0 for l in LABELS],
    dtype=torch.float32,
)
print(f'Class weights: {dict(zip(LABELS, weights.tolist()))}')

## 6. Model ачаалж train хийх

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(str(MODEL_DIR))
model     = AutoModelForAudioClassification.from_pretrained(
    str(MODEL_DIR), num_labels=len(LABELS), ignore_mismatched_sizes=True,
)

train_ds = WebMDataset(train_files, train_labels, extractor)
val_ds   = WebMDataset(val_files,   val_labels,   extractor)

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    remove_unused_columns=False,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    class_weights=weights,
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

## 7. Хадгалах + Drive рүү буцаах

In [ ]:
# Production-той ижил label mapping тавих
model.config.id2label = {i: PROD_MAP[l] for i, l in enumerate(LABELS)}
model.config.label2id = {v: k for k, v in model.config.id2label.items()}
trainer.save_model(str(OUT_DIR))
extractor.save_pretrained(str(OUT_DIR))

# Drive рүү copy
!rm -rf '{OUTPUT_MODEL}'
!cp -r '{OUT_DIR}' '{OUTPUT_MODEL}'

# Шахаж download хийхээр болгох
!cd '{WORK_DIR}' && zip -rq cheating_classifier_v3.zip output_model
!cp '{WORK_DIR}/cheating_classifier_v3.zip' '{DRIVE_BASE}/'

print(f'✓ Хадгалагдсан газрууд:')
print(f'  Drive folder: {OUTPUT_MODEL}')
print(f'  Drive zip:    {DRIVE_BASE}/cheating_classifier_v3.zip')
print(f'\nЛоокал computer-руу татах: Drive-аас cheating_classifier_v3.zip-ийг татаад,')
print(f'  ai-proctor/models/ дотор задлаарай.')

## 8. Validation үр дүн харах

In [ ]:
metrics = trainer.evaluate()
print('Validation metrics:')
for k, v in metrics.items():
    print(f'  {k:25} {v:.4f}')

# Confusion matrix-н оронд дэлгэрэнгүй prediction
from collections import Counter
preds = trainer.predict(val_ds)
pred_labels = preds.predictions.argmax(-1)
true_labels = preds.label_ids

print('\nPer-class accuracy:')
for i, lbl in enumerate(LABELS):
    mask = true_labels == i
    if mask.sum() > 0:
        acc = (pred_labels[mask] == i).mean()
        print(f'  {lbl:12} {acc*100:.1f}%  ({mask.sum()} samples)')